# 🍔 Zomato & Swiggy Analytics - Step 2: Data Cleaning & Feature Engineering

Welcome to the **Data Cleaning & Engineering Pipeline**! In a real-world data analyst role at Swiggy or Zomato, raw transaction logs are often messy, containing missing values, duplicates, out-of-bounds ratings, and negative operational metrics. 

In this notebook, we will use **Pandas** and **Numpy** to clean our raw order dataset and prepare it for SQL analysis and reporting.

### 🎯 Cleaning Checklist:
1. **Remove anomalies**:
   * Null values in critical identifier fields.
   * Duplicate order records.
   * Invalid ratings (stars must be 1 to 5).
   * Negative delivery times.
2. **Standardize columns**:
   * Standardize area names (trim whitespace, title case).
   * Standardize food categories.
   * Standardize time formats.
3. **Create new columns**:
   * `delivery_delay` (latency in minutes beyond standard 30-minute target).
   * `order_hour` (extraction of order time hour).
   * `order_day` (day of week name, e.g., Saturday).

In [ ]:
import pandas as pd
import numpy as np
import os

print("Pandas and Numpy imported successfully!")

## 📥 Step 1: Ingesting the Raw Dataset
Let's load our messy `data/raw_data.csv` file into a Pandas DataFrame and check its starting dimensions.

In [ ]:
raw_path = "../data/raw_data.csv"
if not os.path.exists(raw_path):
    raw_path = "data/raw_data.csv"

df = pd.read_csv(raw_path)
print(f"Initial raw data dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(5)

## 🔍 Step 2: Removing Null Values in Critical Fields
Critical business keys like `order_id`, `customer_id`, or `restaurant_name` must never be null. In our dataset, empty strings or null entries represent incomplete records which we will drop.

**Note**: Operational fields like `delivery_time` and `rating` can legally be null for **Cancelled** orders. However, for completed (**Delivered**) orders, they must be populated. Let's write robust filters.

In [ ]:
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())

# 1. Drop rows where critical IDs are missing or blank
initial_len = len(df)
df = df.dropna(subset=['order_id'])
df = df[df['customer_id'].notna() & (df['customer_id'].astype(str).str.strip() != "")]
df = df[df['restaurant_name'].notna() & (df['restaurant_name'].astype(str).str.strip() != "")]

# 2. Drop rows where Completed (Delivered) orders are missing delivery times or ratings
df = df[~((df['order_status'] == 'Delivered') & (df['delivery_time'].isnull()))]
df = df[~((df['order_status'] == 'Delivered') & (df['rating'].isnull()))]

dropped_nulls = initial_len - len(df)
print(f"\nDropped {dropped_nulls} records due to invalid null values.")
print(f"Current shape: {df.shape}")

## 👥 Step 3: Removing Duplicate Records
Duplicate database writes can skew analytical aggregations (like revenue and total orders). We look for exact duplicate rows and remove them.

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Number of exact duplicate rows identified: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Exact duplicate rows removed successfully.")

print(f"Current shape: {df.shape}")

## ⭐ Step 4: Removing Invalid Ratings
Customer ratings on Swiggy and Zomato are strictly bounded between **1 star** and **5 stars**. Any rating below 1 or above 5 is anomalous and must be filtered out.

In [ ]:
# Convert rating column to numeric, ignoring cancelled blanks
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Filter out completed orders with invalid rating bounds
invalid_ratings = df[(df['order_status'] == 'Delivered') & ((df['rating'] < 1) | (df['rating'] > 5))]
print(f"Identified {len(invalid_ratings)} records with out-of-bounds ratings.")

if len(invalid_ratings) > 0:
    df = df[~((df['order_status'] == 'Delivered') & ((df['rating'] < 1) | (df['rating'] > 5)))]
    print("Out-of-bounds ratings removed.")

print(f"Current shape: {df.shape}")

## ⏱️ Step 5: Removing Negative Delivery Times
Delivery times representing transaction durations must strictly be positive. Let's identify and purge any rows with negative delivery time metrics.

In [ ]:
# Convert delivery_time to numeric, ignoring cancelled blanks
df['delivery_time'] = pd.to_numeric(df['delivery_time'], errors='coerce')

invalid_delivery = df[(df['order_status'] == 'Delivered') & (df['delivery_time'] < 0)]
print(f"Identified {len(invalid_delivery)} records with negative delivery times.")

if len(invalid_delivery) > 0:
    df = df[~((df['order_status'] == 'Delivered') & (df['delivery_time'] < 0))]
    print("Negative delivery times removed.")

print(f"Current shape: {df.shape}")

## 🔠 Step 6: Standardizing Text and Time Formats
To ensure clean SQL aggregations and reporting, we must standardize our features:
1. **Delivery Area Names**: Strip any trailing/leading whitespaces and convert to Title Case (e.g. `"indiranagar  "` -> `"Indiranagar"`).
2. **Food Categories**: Strip whitespaces and title-case them (e.g., `"north indian"` -> `"North Indian"`).
3. **Time Formats**: Convert `order_time` into a unified `datetime` format.

In [ ]:
print("--- Sample of Area names before standardizing ---")
print(df['delivery_area'].unique()[:5])

# 1. Standardize delivery area names
df['delivery_area'] = df['delivery_area'].astype(str).str.strip().str.title()

# 2. Standardize food categories
df['category'] = df['category'].astype(str).str.strip().str.title()

# 3. Standardize time format
df['order_time'] = pd.to_datetime(df['order_time'])

print("\n--- Sample of Area names after standardizing ---")
print(df['delivery_area'].unique())
print("\nTime format standardized: ", df['order_time'].dtypes)

## 🆕 Step 7: Creating Engineered Columns
To help our business teams analyze logistics efficiency, we construct three new columns:
- `delivery_delay`: Subtract standard operational baseline (30 minutes) from `delivery_time` to calculate latency (e.g. `delivery_time - 30`). Negative values signify early deliveries, positive values signify delays. For cancelled orders, this is set to null.
- `order_hour`: Extracted hour of the transaction (0-23).
- `order_day`: Extracted day name of the transaction (e.g. Monday).

In [ ]:
# 1. Create delivery delay column (target time = 30 minutes)
# Orders taking less than 30 mins have a delay of 0, late orders capture positive delay minutes.
df['delivery_delay'] = df['delivery_time'].apply(lambda x: max(0, x - 30) if pd.notnull(x) else np.nan)

# 2. Create order hour column
df['order_hour'] = df['order_time'].dt.hour

# 3. Create order day column (day name)
df['order_day'] = df['order_time'].dt.day_name()

df[['order_time', 'delivery_time', 'delivery_delay', 'order_hour', 'order_day']].head(5)

## 💾 Step 8: Exporting the Cleaned Dataset
We write out the preprocessed dataset to `data/cleaned_data.csv` to keep it ready for downstream sentiment extraction, SQL data modeling, and reporting.

In [ ]:
cleaned_path = "../data/cleaned_data.csv"
if not os.path.exists(os.path.dirname(cleaned_path)):
    cleaned_path = "data/cleaned_data.csv"

df.to_csv(cleaned_path, index=False)
print(f"Data cleaning successfully complete! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")
print(f"Saved dataset to: {cleaned_path}")